<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.24em;text-transform:uppercase;color:#A20025;margin:0 0 12px;">NeurIPS 2026 Education Track&nbsp;&middot;&nbsp;Generate, then verify</div>

<h1 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:33px;line-height:1.12;margin:0 0 16px;color:#1A1416;letter-spacing:-.01em;">Worked solutions 8 to 12. Second opinions, and the transfer</h1>

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:0 0 4px;">

<em style="color:#6B6461;font-size:14.5px;">Full working for X8 to X12. The X10 discussion is long on purpose; it is the second of the two exercises where the learner finds the failure themselves.</em>

Five tasks. X8, X9 and X11 have crisp numerical answers and one written part each.
X10 is the long one. X12 has no key at all, so what follows for it is the rubric and
one fully worked example.

In [1]:
# Path anchor and toolkit import. This is the only setup cell in the notebook.
#
# The bundle looks like this:
#     NEURIPS2026_EDU_GENERATE_VERIFY/
#         gv_toolkit.py          <- the shared toolkit
#         data/                 <- every cached file the exercises read
#         exercises/            <- you are here
#         solutions/
# so the root is one level up from this notebook, and data/ hangs off the root.
# Resolving it positionally means the same cell works locally, on Colab, and in a
# grader's batch run, with no absolute path anywhere.
import sys
from pathlib import Path

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name in {"exercises", "solutions"} else HERE
sys.path.insert(0, str(ROOT))

import gv_toolkit as gv

DATA = gv.DATA
print("bundle root:", ROOT.name, "| cached inputs:", len(gv.REQUIRED_FILES) + len(gv.SUPPORTING_FILES))

bundle root: NEURIPS2026_EDU_GENERATE_VERIFY | cached inputs: 25


In [2]:
import json
import numpy as np
from rdkit import Chem, RDLogger
from scipy.stats import spearmanr

RDLogger.DisableLog("rdApp.*")

<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:34px 0 10px;color:#1A1416;"><span style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:13px;color:#A20025;font-weight:500;margin-right:12px;">&sect;8</span>X8. Did the scaffold survive</h2>

In [3]:
inpaint = json.loads((DATA / "diffsbdd_inpaint_metrics.json").read_text())
records = inpaint["records"]
median_coord_retention = float(np.median([r["scaffold_coord_retention"]
                                          for r in records]))

PURINE = Chem.MolFromSmarts("c1ncc2ncnc2n1")
inpainted = gv.read_sdf(DATA / "diffsbdd_inpainted.sdf")
n_smarts_keep = sum(1 for m in inpainted if m.HasSubstructMatch(PURINE))
smarts_retention_pct = 100.0 * n_smarts_keep / len(inpainted)

print(f"by coordinate: median retention {median_coord_retention}")
print(f"               all ten at 1.0:  "
      f"{all(r['scaffold_coord_retention'] == 1.0 for r in records)}")
print(f"by SMARTS:     {n_smarts_keep} of {len(inpainted)} "
      f"= {smarts_retention_pct:.1f} percent")
print(f"the file's own flag agrees:    "
      f"{sum(1 for r in records if r['keeps_purine_core'])} of {len(records)}, "
      f"reported as {inpaint['scaffold_retention_pct']} percent")

by coordinate: median retention 1.0
               all ten at 1.0:  True
by SMARTS:     2 of 10 = 20.0 percent
the file's own flag agrees:    2 of 10, reported as 20.0 percent


**By coordinate: 1.00.** All ten molecules kept the nine pinned purine atoms exactly
where they were placed. The median is 1.0 and so is every individual value.

**By SMARTS: 2 of 10, a 20 percent retention rate.** A strict aromatic-purine pattern
matches only two of the ten resulting molecules.

**The difference is not a failure of inpainting, and this is the whole task.**
Inpainting fixes two things: atom positions and element identities. It does not fix
bonds, because a diffusion model over 3D molecular structure of this kind does not
carry bond orders as state. The bonds are re-perceived from the final geometry
afterwards, by OpenBabel in this pipeline, and bond perception is a heuristic that
looks at interatomic distances and hybridization and decides where the double bonds
and the aromatic rings go. A nine-atom ring system whose atoms are in exactly the
right places can still come back with a different aromatic-versus-aliphatic
assignment, and then a SMARTS pattern written with lowercase aromatic atoms stops
matching.

So the two numbers measure different things. The coordinate measurement answers the
question the method promises to answer: did the atoms you pinned stay pinned. Yes,
all of them. The SMARTS measurement answers a downstream question about what a
cheminformatics toolkit made of the result, and it inherits every convention of the
perception step. Reporting 20 percent as "the scaffold retention rate" without saying
which measurement it is would badly misrepresent the method.

Notice that this is the same move as X3 and X6. A single-number summary is defensible
only alongside the definition that produced it, and here two defensible definitions
of the same word differ by a factor of five.

**The fourth task.** Any answer that correctly names the mask and the chain start
earns the mark. Two that work:

Fix the pocket and a subset of the ligand's atoms chosen by a medicinal chemist
rather than by an automatic scaffold definition, starting the chain from pure noise
on the unmasked atoms. This is the identical operation with a hand-drawn mask, and it
is what a project wants when a chemist says "keep this hinge binder, change
everything else".

Or fix the pocket and start the chain from a noised known inactive rather than from
noise, running only partway back up the ladder. That gives you a local exploration
around a molecule you want to move away from, which is the same shape of operation as
the QED optimization run but with a different seed and a different objective.

The mark is for naming what is held fixed and where the reverse chain starts, not for
originality.

**On the published defect.** The shipped QED-optimization cell ends with a
print that reads a variable assigned only on the fresh-compute branch, so running the
shipped cached path raises `NameError` after the figure has already rendered. The
figure and every plotted value are correct; only the trailing line fails. This
exercise reads `best_qed` from `diffsbdd_optimize_metrics.json` (0.7229067081355963)
rather than reusing that print, and says so in the exercise notebook rather than
silently patching it. A learner who can see a real published defect and the one-line
reason for it learns more than a learner handed a file where it never happened.

<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:34px 0 10px;color:#1A1416;"><span style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:13px;color:#A20025;font-weight:500;margin-right:12px;">&sect;9</span>X9. Ask a second method</h2>

In [4]:
dd = gv.read_metrics_csv(DATA / "diffdock_redock_posebusters.csv")
pb = gv.read_metrics_csv(DATA / "posebusters_diffsbdd.csv")

n_within_2_top = int((dd.rmsd_generated_vs_redocked_A < 2).sum())
n_within_2_any = int((dd.min_rmsd_any_pose_A < 2).sum())
median_rmsd = float(dd.rmsd_generated_vs_redocked_A.median())
n_centroid_within_5 = int((dd.redock_centroid_dist_to_caffeine_A < 5).sum())
median_centroid = float(dd.redock_centroid_dist_to_caffeine_A.median())

print(f"within 2 A, top pose:            {n_within_2_top} of {len(dd)}")
print(f"within 2 A, best of ten poses:   {n_within_2_any} of {len(dd)}")
print(f"median generated-vs-redocked:    {median_rmsd:.2f} A")
print(f"re-docked centroids within 5 A:  {n_centroid_within_5} of {len(dd)}")
print(f"median re-docked centroid dist:  {median_centroid:.2f} A")

# The same quantity for the GENERATED poses, recomputed from the structures.
caffeine = gv.read_sdf(DATA / "3rfm_caffeine_ref.sdf")[0]
caf_centroid = caffeine.GetConformer().GetPositions().mean(0)
gen_centroids = [float(np.linalg.norm(m.GetConformer().GetPositions().mean(0)
                                      - caf_centroid))
                 for m in gv.read_sdf(DATA / "diffsbdd_generated.sdf")]
print(f"median GENERATED centroid dist:  {float(np.median(gen_centroids)):.3f} A")

within 2 A, top pose:            0 of 20
within 2 A, best of ten poses:   2 of 20
median generated-vs-redocked:    4.39 A
re-docked centroids within 5 A:  19 of 20
median re-docked centroid dist:  1.77 A
median GENERATED centroid dist:  0.632 A


In [5]:
CHECKS = ["bond_lengths", "bond_angles", "minimum_distance_to_protein"]
for c in CHECKS:
    print(f"{c:<32s} generated {pb[c].mean() * 100:5.0f}%   "
          f"re-docked {dd[c].mean() * 100:5.0f}%")

n_valid_generated = int(pb.pb_valid.sum())
n_valid_redocked = int(dd.pb_valid.sum())
mindist_fail_pct = float((1 - dd.minimum_distance_to_protein.mean()) * 100)
print(f"\nfully valid: {n_valid_generated} generated, {n_valid_redocked} re-docked")
print(f"re-docked poses failing minimum_distance_to_protein: {mindist_fail_pct:.0f}%")
print(f"DiffDock median top-pose confidence: "
      f"{float(dd.diffdock_top_confidence.median()):.4f}, negative for "
      f"{int((dd.diffdock_top_confidence < 0).sum())} of {len(dd)}")

bond_lengths                     generated    90%   re-docked   100%
bond_angles                      generated    85%   re-docked   100%
minimum_distance_to_protein      generated   100%   re-docked    45%

fully valid: 15 generated, 8 re-docked
re-docked poses failing minimum_distance_to_protein: 55%
DiffDock median top-pose confidence: 0.0971, negative for 7 of 20


**The three answers.** Zero of twenty top-ranked re-docked poses land within 2
angstroms of the generated pose, with a median separation of 4.39 angstroms.
Allowing the closest of the ten poses DiffDock returns per molecule, two of twenty.
Nineteen of twenty re-docked centroids sit within 5 angstroms of the co-crystallized
caffeine, with a median of 1.77 angstroms; the generated poses have a median centroid
distance of 0.632 angstroms, which is recomputed above from the shipped structures
rather than read off a table.

Quote 0.63 for that last figure. The original figure prints 0.6; 0.63 is the value
measured from the shipped structures.

**On the battery, the two sources fail in opposite directions.** Re-docked poses pass
bond lengths and bond angles at 100 percent and violate
`minimum_distance_to_protein` on 55 percent of molecules. Generated poses pass every
ligand-to-protein check and fail bond angles on 15 percent and bond lengths on 10
percent. Fully valid: 15 of 20 generated against 8 of 20 re-docked.

That pattern is exactly what the two methods' training objectives predict, and it is
worth spelling out. DiffSBDD generates a molecule and its pose jointly while
conditioned on the pocket, so it has learned where the pocket is and has not
especially learned bond geometry. DiffDock places an existing rigid-ish molecule, so
the internal geometry it was handed is preserved perfectly while the placement is
free to drift into the protein. Neither is a bug; each method is good at the thing it
was trained on and indifferent to the thing it was not.

**The sentence.** Two independent methods agree about which site the molecule
occupies and disagree about the pose within it by about 4 angstroms, so a single
rendered pose deserves confidence about the site and very little about the geometry.

**The trap.** The disagreement is not evidence that one method is wrong. There is no
experimental structure for a molecule that has never been synthesized, so neither
pose can be called correct. What you have is two estimates and their separation, and
the separation is the useful quantity: it puts a floor under your uncertainty about
the pose, computed without any ground truth at all. That is what a second opinion
buys you when no ground truth exists, and it is the same thing scRMSD buys you in
X11.

Worth showing the learner: DiffDock's own median top-pose confidence is 0.097, and
it is negative for seven of the twenty. The docking model is hedging on its own
answers, in writing, before anyone compares it to anything.

<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:34px 0 10px;color:#1A1416;"><span style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:13px;color:#A20025;font-weight:500;margin-right:12px;">&sect;10</span>X10. Read the wrong number on purpose</h2>

The second failure-discovery exercise, and the one that produces the loudest reaction
in a live session.

In [6]:
bz = gv.read_metrics_csv(DATA / "boltz2_comparison.csv")

# Part one. The conversion the stem asks for, before the stem tells you anything.
caffeine_row = bz[bz.mol_id == "caffeine_control"].iloc[0]
flowmol_row = bz[bz.mol_id == "flowmol3_ref"].iloc[0]
caffeine_uM = 10 ** float(caffeine_row.modal_affinity_pred_value_log10_IC50_uM)
flowmol_uM = 10 ** float(flowmol_row.modal_affinity_pred_value_log10_IC50_uM)
print(f"caffeine:           {caffeine_uM:.1f} micromolar")
print(f"FlowMol3 molecule:  {flowmol_uM:.1f} micromolar   <- INVALID, see part three")
print(f"\nread at face value, that says the generated molecule is "
      f"{caffeine_uM / flowmol_uM:.1f}x more potent than the drug that binds.")

caffeine:           25.0 micromolar
FlowMol3 molecule:  13.3 micromolar   <- INVALID, see part three

read at face value, that says the generated molecule is 1.9x more potent than the drug that binds.


In [7]:
# Part two. The other head.
print("binder probability, pinned local build:")
for _, r in bz.sort_values("modal_affinity_probability_binary",
                           ascending=False).iterrows():
    flag = "above the threshold" if r.modal_affinity_probability_binary > 0.5 else ""
    print(f"  {r.mol_id:<20s} {r.modal_affinity_probability_binary:.4f}  {flag}")

n_above_threshold = int((bz.modal_affinity_probability_binary > 0.5).sum())
print(f"\n{n_above_threshold} of {len(bz)} molecules clear the 0.5 binary threshold: "
      "caffeine, the co-crystal positive control.")
print(f"the FlowMol3 molecule scores "
      f"{float(flowmol_row.modal_affinity_probability_binary):.4f}")

binder probability, pinned local build:
  caffeine_control     0.6219  above the threshold
  diffsbdd_3rfm_0      0.4440  
  diffsbdd_3rfm_2      0.4391  
  diffsbdd_3rfm_16     0.1851  
  diffsbdd_3rfm_18     0.0911  
  flowmol3_ref         0.0906  
  diffsbdd_3rfm_4      0.0866  

1 of 7 molecules clear the 0.5 binary threshold: caffeine, the co-crystal positive control.
the FlowMol3 molecule scores 0.0906


### Parts one to three

**Part one.** Caffeine's continuous value of 1.3978 gives 25.0 micromolar. The
FlowMol3 molecule's 1.1239 gives 13.3 micromolar. Read at face value, an
unconditionally generated molecule that has never existed is twice as potent as the
drug this receptor is co-crystallized with.

**Part two.** Caffeine 0.6219. The DiffSBDD designs 0.4440, 0.4391, 0.1851, 0.0911
and 0.0866. The FlowMol3 molecule 0.0906. Exactly one of the seven clears the 0.5
binary threshold, and it is the positive control.

**Part three, what is wrong.** The continuous head and the binary head answer
different questions and are calibrated on different populations. The binary head asks
whether this is a binder at all. The continuous head asks how potent it is, and it is
trained on molecules already known to be active. For a molecule the binary head has
placed at 0.09, the continuous head is being asked to extrapolate a potency for
something it has just said is probably not a binder. It will return a number, because
that is what a regression head does. The number means nothing.

So the 13.3 micromolar figure is not a small error or an optimistic estimate. It is
undefined, and the correct handling is to read the probability first and never
perform the conversion at all. **That number must never be quoted as a potency,
anywhere, including in this notebook, where it appears only as the thing you compute
and then discard.**

The order of the parts is deliberate and it is the pedagogical device. You were asked
to do the conversion before being told it was invalid, so the absurd conclusion
arrived in your own output rather than in a warning box you would have skimmed. A
warning you have already violated is much harder to forget.

**One conversion here is legitimate.** Caffeine's 25 micromolar is defensible,
because caffeine is above the threshold, and it is worth checking that it is also roughly
right: caffeine's measured affinity at the adenosine A2A receptor is in the tens of
micromolar. Say it is a model estimate rather than a measurement, and the sentence
stands.

In [8]:
# Part four. Two implementations of one model.
rho = float(spearmanr(bz.modal_affinity_probability_binary,
                      bz.nim_affinity_probability_binary).statistic)
mean_abs_delta = float((bz.nim_affinity_probability_binary
                        - bz.modal_affinity_probability_binary).abs().mean())
n_ranks_identical = int((bz.modal_rank == bz.nim_rank).sum())

print(f"Spearman rho:                        {rho:.4f}")
print(f"mean absolute probability delta:     {mean_abs_delta:.4f}")
print(f"identical ranks:                     {n_ranks_identical} of {len(bz)}")
print()
print("every summary statistic says the two implementations agree.")
print()
moved = bz[bz.modal_rank != bz.nim_rank]
print("and yet:")
print(moved[["mol_id", "modal_affinity_probability_binary",
             "nim_affinity_probability_binary", "modal_rank",
             "nim_rank"]].to_string(index=False))
print()
p0 = bz[bz.mol_id == "diffsbdd_3rfm_0"].iloc[0]
p2 = bz[bz.mol_id == "diffsbdd_3rfm_2"].iloc[0]
print(f"pinned build:  design 0 leads design 2 by "
      f"{p0.modal_affinity_probability_binary - p2.modal_affinity_probability_binary:.5f}")
print(f"hosted build:  design 2 leads design 0 by "
      f"{p2.nim_affinity_probability_binary - p0.nim_affinity_probability_binary:.5f}")

Spearman rho:                        0.9643
mean absolute probability delta:     0.0290
identical ranks:                     5 of 7

every summary statistic says the two implementations agree.

and yet:
         mol_id  modal_affinity_probability_binary  nim_affinity_probability_binary  modal_rank  nim_rank
diffsbdd_3rfm_0                           0.443957                         0.363281           2         3
diffsbdd_3rfm_2                           0.439090                         0.449219           3         2

pinned build:  design 0 leads design 2 by 0.00487
hosted build:  design 2 leads design 0 by 0.08594


### Part four, and why it is the sharper half

**The numbers.** Spearman rho 0.9643 over the seven molecules. Mean absolute
probability difference 0.029. Five of seven ranks identical. Every one of those is
the kind of figure a methods section prints to establish that two implementations
agree, and every one of them is true.

The two that swap are designs 0 and 2. They are the top two designs. Under the pinned
local build, design 0 leads at 0.4440 against 0.4391, a gap of 0.0049. Under the
hosted service the order reverses and design 2 leads at 0.4492 against 0.3633, a gap
of 0.0859.

**What that does to the sentence.** "Design 2 was our best candidate" is not a
statement about the molecules and it is not a statement about Boltz-2. It is a
statement about which implementation of Boltz-2 was run, and nothing in the sentence
records that. Swap the implementation and the sentence becomes false, while every
aggregate statistic in the paper stays the same.

This is worth sitting with, because it inverts the usual intuition about
reproducibility. The failure mode people expect is two implementations that disagree
badly, which is loud and gets caught. What happened here is two implementations that
agree almost perfectly, which is quiet and does not get caught, and the disagreement
falls precisely on the one claim a project does make. Nobody publishes "the mean
absolute probability delta between implementations was 0.029". Everybody publishes
"design 2 was our best candidate".

**The general form.** A summary statistic averages over the thing you care about. If
your decision depends on a ranking near the top, then rank correlation over the whole
list is the wrong diagnostic, because it is dominated by the agreement among the
molecules you were never going to choose. The right diagnostic is agreement on the
decision itself: does the same design come out on top. Here the answer is no, and the
margin separating the top two is 0.0049 under one build, which is smaller than the
0.029 mean disagreement between the builds. The ranking was never resolvable at that
precision, and the summary statistics were not the place that would tell you.

**What survives.** Both implementations agree that exactly one of the seven molecules
is above the threshold, and that it is caffeine. Both put the same four molecules at the
bottom. The plausibility question, "which of these are worth carrying forward", is
answered identically by both. Only "which one is best" is implementation-dependent.
That is the distinction to carry out of this exercise: a verification result can be
stable at the granularity of a shortlist and fragile at the granularity of a winner,
and which granularity you need depends on what you are about to do with it.

### What the learner should now distrust

**A single top-ranked candidate, from any scoring model, without a stability check.**
The cheapest version of that check is the one you just ran: score the shortlist twice
under different implementations, seeds or versions, and see whether the winner is
stable. If the top two are separated by less than the run-to-run variation, you have
a shortlist, not a winner, and you should say so.

**A rank correlation offered as evidence that two pipelines agree.** Ask what
decision the pipeline output feeds, and demand the agreement statistic for that
decision. Rho of 0.96 across seven molecules is compatible with the top two being
swapped, which the number above demonstrates rather than argues.

**Any single number from a multi-head model, read without its companion.** The
threshold head and the magnitude head have to be read in order. This generalizes past Boltz-2:
a confidence and a prediction, an applicability-domain flag and a potency, an
uncertainty estimate and a mean. When a model gives you two outputs, one of them is
usually telling you whether to believe the other.

### The question to ask of any verification result you meet in a paper

**Which decision does this number support, and is the number stable at the precision
that decision requires?**

For a reported best candidate: what was the margin over second place, and what is the
run-to-run variation in the score. For a reported correlation: is the correlation
over the whole population, when the claim is about the top of it. For a reported
affinity: what was the binder probability, and was the affinity head even defined for
that molecule.

<ol style="font-size:13.5px;color:#6B6461;padding-left:20px;">
<li style="margin:7px 0;">Passaro S. et al. (2025). Boltz-2: Towards Accurate and Efficient Binding Affinity Prediction. bioRxiv. <a href="https://doi.org/10.1101/2025.06.14.659707">doi:10.1101/2025.06.14.659707</a>. The model whose two affinity heads this exercise reads in the wrong order and then the right one.</li>
</ol>

<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:34px 0 10px;color:#1A1416;"><span style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:13px;color:#A20025;font-weight:500;margin-right:12px;">&sect;11</span>X11. The same test, a different molecule</h2>

In [9]:
sc = json.loads((DATA / "selfconsistency_full.json").read_text())
designs = sc["all"]
n_designable = sum(1 for d in designs if d["scrmsd"] < 2.0 and d["plddt"] > 70)
median_scrmsd = float(np.median([d["scrmsd"] for d in designs]))
print(f"designable: {n_designable} of {len(designs)}")
print(f"median scRMSD: {median_scrmsd}")
worst = max(designs, key=lambda d: d["scrmsd"])
print(f"the one exception grazes the box at scRMSD {worst['scrmsd']} "
      f"(pLDDT {worst['plddt']})")

scaling = json.loads((DATA / "designability_scaling.json").read_text())
per_design = scaling["per_design"]
designable_fraction = {}
for L in scaling["lengths"]:
    at_L = [d for d in per_design if d["length"] == L]
    ok = sum(1 for d in at_L if d["scrmsd"] < 2.0 and d["plddt"] > 70)
    designable_fraction[L] = round(100.0 * ok / len(at_L), 1)
    print(f"  {L:>3} residues   {ok:>2} of {len(at_L)}   "
          f"{designable_fraction[L]:>5.1f} percent")

designable: 23 of 24
median scRMSD: 0.715
the one exception grazes the box at scRMSD 2.05 (pLDDT 84.4)
   60 residues   14 of 24    58.3 percent
   90 residues   17 of 24    70.8 percent
  120 residues   20 of 24    83.3 percent
  150 residues   21 of 24    87.5 percent


**The numbers.** Twenty-three of 24 designs sit inside the designable box; the single
exception grazes it at 2.05 angstroms with a pLDDT that would otherwise have passed.
Median scRMSD is 0.715 angstroms. Designable fraction by backbone length: 58.3
percent at 60 residues, 70.8 at 90, 83.3 at 120, 87.5 at 150.

**Does it degrade with length?** No. It rises. That is the opposite of the intuition
most people bring, which is that longer things are harder, and the explanation is
that at 60 residues there is less secondary structure for ProteinMPNN to design
against and less for ESMFold to recognize, so a short backbone is a harder target for
the whole pipeline rather than an easier one. Worth flagging in a seminar as an
example of an assumption that a five-line calculation overturns.

**The transfer sentence, which is the graded part.** RFdiffusion plays the generator,
the role DiffSBDD played in modules 5 and 6. ESMFold plays the independent second
method, the role DiffDock played in X9. ProteinMPNN is the bridge that turns a
backbone into something the second method can take as input, which docking did not
need because a small molecule is already its own input. And the scRMSD is the same
kind of quantity as the pose RMSD in X9: a disagreement between two methods that
never saw each other's output, used as a proxy for correctness in a setting where no
ground truth exists.

**Why this is the most diagnostic item in the set.** Every other exercise can be
completed by executing a procedure. This one cannot. A learner who has memorized two
pipelines will write something about proteins and something about ligands and will
not be able to say which model plays which role, because the roles only exist if you
have understood the loop as a loop. A learner who has understood it will write the
sentence in thirty seconds. Mark it strictly: the answer has to name the generator,
the second method, and the disagreement quantity, and it has to map each one onto its
counterpart in X9.

<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:34px 0 10px;color:#1A1416;"><span style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:13px;color:#A20025;font-weight:500;margin-right:12px;">&sect;12</span>X12. Read a paper you have not seen</h2>

No key, because the answer depends on the paper. Here is the rubric and one fully
worked example.

**The rubric, five points.** One point each for correctly naming the generator, the
verifier and the reported metric. One point for naming a module of this curriculum
that teaches that metric. One point for a missing-information question that
is answerable and specific. Three of five is a pass.

Reject "more detail would be helpful" and anything else that could be written without
having read the abstract. Accept anything that a methods section could answer.

**A worked example, for FlowR (Cremer et al., 2026).**

Generator: a structure-conditioned flow-matching model over continuous coordinates
and categorical atom types, using equivariant optimal transport to couple the noise
and data distributions. It is the same family as the FlowMol3 model in module 3, with
a receptor in the conditioning.

Verifier: PoseBusters for physical validity, pose accuracy against a reference, and
interaction recovery against the reference complex.

Reported numbers: PoseBusters validity as a percentage, pose RMSD, and the fraction
of reference interactions recovered.

Which module prepared you: modules 3 and 4 prepared the validity number, because you
now know that a PoseBusters percentage depends on the receptor preparation and that
several of its checks may be vacuous. Module 6 prepared the pose-accuracy number,
because you have seen two methods disagree by 4 angstroms on the same molecule with
neither being wrong.

What the abstract does not tell you, and you would now check: how the receptor was
prepared, specifically whether HETATM records were retained and which, because X5
showed that this single choice moves the validity verdict from 15 of 20 to 0 of 20.
Second: what defines a recovered interaction, and under which tool's definition,
because X6 and X7 showed that a similarity-style number is uninterpretable without
its definition.

**Marking note.** The four papers listed in the exercise are the same set cited in
the resource's own reference list, which is deliberate: the exercise sends learners
to the frontier work the curriculum claims to prepare them for, so the claim is
checkable by the learner rather than asserted by the author.

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">Answer summary</h2>

X8: coordinate retention 1.00 for all ten; SMARTS retention 2 of 10, 20.0 percent;
the difference is bond re-perception after inpainting, which fixes positions and
elements but not bond orders; best optimized QED 0.7229.

X9: zero of twenty within 2 angstroms on the top pose and two of twenty allowing any
returned pose; median separation 4.39 angstroms; nineteen of twenty re-docked
centroids within 5 angstroms of caffeine at a median of 1.77, against 0.63 for the
generated poses; 15 of 20 generated fully valid against 8 of 20 re-docked, failing in
opposite directions.

X10: caffeine 25.0 micromolar and the FlowMol3 molecule 13.3, the second of which is
invalid and must never be quoted; one of seven above the 0.5 threshold; Spearman 0.9643,
mean absolute delta 0.029, five of seven ranks identical, and yet designs 0 and 2,
the top two, swap between implementations.

X11: 23 of 24 designable, median scRMSD 0.715; designable fraction 58.3, 70.8, 83.3
and 87.5 percent at 60, 90, 120 and 150 residues, so it rises with length;
RFdiffusion is the generator, ESMFold the independent second method, scRMSD the
analogue of pose RMSD.

X12: rubric-graded, three of five to pass.